# Lesson 1.4 — Multi-Agent Systems and Orchestration

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Compare orchestration topologies (single agent, orchestrator-worker, parallel)
- Decide when multiple agents add value versus added failure surface
- Sketch an orchestrator-worker design with explicit handoff contracts

**You will produce:** An orchestrator-worker design with typed handoff contracts between agents.

## Concepts

Multiple agents help when subtasks are **separable and benefit from specialization or parallelism** — e.g. an orchestrator that fans work out to focused workers (researcher, writer, checker) and synthesizes results. Common topologies:

- **Orchestrator-worker** — a lead agent decomposes and delegates; workers return structured results. Good for research/synthesis.
- **Parallel (sectioning/voting)** — independent calls run concurrently, then aggregate. Good for speed or ensembling.

Each added agent multiplies cost, latency, and failure modes, and makes evaluation harder. The architect's job is to justify *why* coordination beats one well-prompted call. Define **handoff contracts** (the exact structured payload passed between agents) so the system stays debuggable.

> **Reference:** [Anthropic — Building effective agents (orchestrator-workers)](https://docs.anthropic.com/en/docs/build-with-claude/building-effective-agents)

In [ ]:
# !pip install anthropic
# Guarded Claude client setup — every lesson runs with or without an API key.
import os

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic()
        LIVE = True
    else:
        client = None
        LIVE = False
except Exception as e:  # SDK not installed in this kernel
    client = None
    LIVE = False
    print("anthropic SDK unavailable:", e)

# Current Claude model family (confirm exact IDs at https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = {
    "opus": "claude-opus-4-20250514",    # deepest reasoning, highest cost/latency
    "sonnet": "claude-sonnet-4-20250514",# balanced default workhorse
    "haiku": "claude-3-5-haiku-20241022",# fastest/cheapest for simple tasks
}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    """Live call when a key exists; otherwise a deterministic mock so notebooks run."""
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system or "You are a helpful solution architect.",
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Define handoff contracts between agents

The contract — not the prompt — is the real interface. We model worker outputs as typed dataclasses the orchestrator can validate before synthesizing.

In [ ]:
from dataclasses import dataclass, asdict
from typing import List

@dataclass
class WorkerResult:
    agent: str
    findings: List[str]
    citations: List[str]
    confidence: float  # 0..1

def validate_handoff(r: WorkerResult):
    assert 0.0 <= r.confidence <= 1.0, "confidence out of range"
    assert r.findings, "empty findings not allowed"
    return True

demo = WorkerResult("policy-researcher", ["Plan covers MRI with prior auth"], ["policy_7.pdf#p3"], 0.82)
print(validate_handoff(demo), asdict(demo))

## Code-Along: Step 2 — Model an orchestrator that fans out and synthesizes

The orchestrator delegates to specialized workers (here mocked) and refuses to synthesize low-confidence results — a built-in quality gate.

In [ ]:
def worker(agent, prompt):
    text = ask_claude(prompt, system=f"You are the {agent} agent. Return concise findings.")
    # In production you'd parse structured output; we mock the contract here.
    return WorkerResult(agent, [text[:60]], [f"{agent}_src"], 0.8)

def orchestrate(case):
    workers = [
        worker("policy-researcher", f"Find policy rules for: {case}"),
        worker("clinical-checker", f"Check clinical criteria for: {case}"),
    ]
    good = [w for w in workers if validate_handoff(w) and w.confidence >= 0.7]
    if not good:
        return "ESCALATE: no confident worker result"
    joined = "; ".join(f for w in good for f in w.findings)
    return ask_claude(f"Synthesize a recommendation from: {joined}",
                      system="You are the orchestrator. Be decisive and cite workers.")

print(orchestrate("MRI prior auth for member 123"))

## Exercise

Add a third worker (`risk-auditor`) and change `orchestrate` to run a **parallel voting** step: if two of three workers disagree on approve/deny, escalate to a human instead of synthesizing. Model the vote however you like (e.g. a label on each `WorkerResult`).

In [ ]:
# TODO: extend WorkerResult with a 'recommendation' label (approve/deny/needs_info)
# TODO: add a risk-auditor worker
# TODO: implement majority voting; escalate on disagreement

## Key Takeaways

- Use multiple agents only when subtasks are separable and benefit from specialization or parallelism.
- The handoff contract is the real interface — type and validate it.
- Build quality gates (confidence thresholds, voting) into the orchestrator.
- Every added agent multiplies cost, latency, and failure surface — justify it.

---

**Next:** [Lesson 1.5 — Decomposition and the Architecture Decision Record](./lesson-1.5-decomposition-and-adr.ipynb)